# VL-JEPA: Vision-Language Joint Embedding Predictive Architecture

**Course:** Advanced Deep Learning | **Topic:** Vision-Language Models  
**Paper:** VL-JEPA (arXiv:2512.10942v1)

## Overview

This notebook implements VL-JEPA, a **non-generative** vision-language model that learns aligned representations through prediction rather than generation. Unlike generative models (e.g., CLIP, BLIP), VL-JEPA predicts embeddings of text descriptions directly from visual input, avoiding the computational overhead of token-by-token generation.

**Key architectural principles:**
1. **Joint Embedding Predictive Architecture (JEPA)**: Predict target representations in embedding space rather than pixel/token space
2. **Frozen vision encoder**: Leverage pre-trained visual representations (V-JEPA2) without fine-tuning
3. **Contrastive learning**: Align vision and language modalities using InfoNCE loss
4. **Non-causal attention**: Unlike autoregressive LLMs, the predictor attends bidirectionally across the sequence

## Multi-GPU Configuration

**GPU allocation for memory efficiency:**
- **GPU 0**: V-JEPA2 vision encoder (frozen, 304M params)
- **GPU 1**: Llama predictor (490M params) + Gemma text encoder (2B params) + projection heads

**Data flow:**
```
Disk → CPU → GPU 0 (vision) → GPU 1 (predictor + text) → Loss
```

Dataset streams from disk (memory-mapped) to minimize RAM usage. Only current batch resides in VRAM.

In [ ]:
%pip install torch torchvision torchaudio transformers datasets huggingface-hub

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoVideoProcessor, LlamaForCausalLM
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import numpy as np

# Check GPU availability
assert torch.cuda.device_count() >= 2, 'Need at least 2 GPUs'
device0 = torch.device('cuda:0')
device1 = torch.device('cuda:1')
print(f'GPU 0: {torch.cuda.get_device_name(0)} - {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'GPU 1: {torch.cuda.get_device_name(1)} - {torch.cuda.get_device_properties(1).total_memory/1e9:.1f}GB')

## 1. Vision Encoder: V-JEPA2 (GPU 0)

**Model:** V-JEPA2 ViT-L (Vision Transformer Large)  
**Pre-training:** Self-supervised video prediction (JEPA objective)  
**Role:** Extract spatiotemporal features from video input

**Why V-JEPA2?**
- Pre-trained on video prediction task → captures temporal dynamics
- Produces semantically rich embeddings without requiring labeled data
- Frozen during training → acts as fixed feature extractor (computational efficiency)

**Input format:** `[B, T, C, H, W]` (batch, time, channels, height, width)  
V-JEPA2 is a video model, so even for static images we create a temporal dimension by repeating frames.

**Design decision:** Keeping the vision encoder frozen reduces memory requirements and prevents catastrophic forgetting of visual features. The predictor learns to map these frozen representations to language space.

In [ ]:
print('Loading V-JEPA2 vision encoder on GPU 0...')

model_id = 'facebook/vjepa2-vitl-fpc64-256'
vision_processor = AutoVideoProcessor.from_pretrained(model_id)
vision_model = AutoModel.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device0)

print(f'✓ Loaded V-JEPA2: {model_id}')

# Freeze
for param in vision_model.parameters():
    param.requires_grad = False
vision_model.eval()

vision_dim = vision_model.config.hidden_size
print(f'Hidden dim: {vision_dim}')
print(f'Params: {sum(p.numel() for p in vision_model.parameters())/1e6:.1f}M (frozen, on GPU 0)')

## 2. Predictor: Llama-3.2-1B Layers (GPU 1)

**Architecture:** Last 8 Transformer layers (layers 8-15) from Llama-3.2-1B  
**Role:** Transform visual embeddings to predict text embeddings

**Why Llama layers?**
- Pre-trained on language data → understands semantic structure
- Transformer self-attention → models relationships between visual tokens
- **Critical modification:** Non-causal attention mask (see Section 3.1 of VL-JEPA paper)

### Non-Causal vs. Causal Attention

**Causal attention (standard LLMs):** Token at position *i* can only attend to positions ≤ *i*
- Required for autoregressive generation (predicting next token)
- Attention mask is lower-triangular

**Non-causal attention (VL-JEPA):** Token at position *i* can attend to ALL positions
- Required for joint encoding of visual and query embeddings
- Attention mask is all ones (no masking)

**Why non-causal?** We're not generating a sequence left-to-right. We're encoding a complete visual input to predict a complete text embedding. Bidirectional context improves representational quality.

**Implementation detail:** We pass `cache_position` for Rotary Position Embeddings (RoPE), which encode positional information without requiring causal masking.

In [ ]:
print('\nLoading Llama-3.2-1B on GPU 1...')
llama = LlamaForCausalLM.from_pretrained(
    'meta-llama/Llama-3.2-1B',
    torch_dtype=torch.float16
).to(device1)

# Use full model but freeze first 8 layers (0-7)
predictor = llama.model  # Use the full LlamaModel
predictor_dim = llama.config.hidden_size

# Freeze embedding and first 8 layers
for param in predictor.embed_tokens.parameters():
    param.requires_grad = False
for i in range(8):
    for param in predictor.layers[i].parameters():
        param.requires_grad = False

# Keep layers 8-15 trainable
trainable_params = sum(p.numel() for p in predictor.layers[8:16].parameters())
print(f'✓ Using full Llama model')
print(f'  Frozen: layers 0-7 + embeddings')
print(f'  Trainable: layers 8-15 ({trainable_params/1e6:.1f}M params)')
print(f'Hidden dim: {predictor_dim}')

# Input projection (GPU 1) - initialize in float32 first for stability
predictor_proj_in = nn.Linear(vision_dim, predictor_dim)
# Initialize weights properly before moving to GPU and converting to float16
nn.init.xavier_uniform_(predictor_proj_in.weight)
nn.init.zeros_(predictor_proj_in.bias)
# Now move to device and convert
predictor_proj_in = predictor_proj_in.to(device1).half()
print(f'Projection: {vision_dim} → {predictor_dim} (on GPU 1)')

## 3. Text Encoder: Gemma-2-2B (GPU 1)

**Architecture:** Gemma-2-2B (Google's open LLM)  
**Role:** Encode text descriptions into target embeddings (Y-encoder in JEPA terminology)

**Why Gemma?**
- Strong language understanding from pre-training
- Produces contextual embeddings that capture semantic meaning
- 2B parameters provide sufficient capacity without excessive memory usage

**Training regime:** Fine-tuned with reduced learning rate (0.05× base LR)
- Prevents catastrophic forgetting of language knowledge
- Allows gentle adaptation to vision-language alignment task

**Role in architecture:**
- **Predictor (Llama)** learns to predict what the text embedding should be
- **Text encoder (Gemma)** provides the ground truth target embeddings
- Contrastive loss pulls predicted and target embeddings together in joint space

In [ ]:
print('\nLoading Gemma-2-2B on GPU 1...')
text_model = AutoModel.from_pretrained(
    'google/gemma-2-2b',
    torch_dtype=torch.float16
).to(device1)
text_tokenizer = AutoTokenizer.from_pretrained('google/gemma-2-2b')

if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
    print('Set pad_token = eos_token')

text_dim = text_model.config.hidden_size
print(f'✓ Loaded Gemma')
print(f'Hidden dim: {text_dim}')
print(f'Params: {sum(p.numel() for p in text_model.parameters())/1e6:.1f}M (on GPU 1)')

## 4. Dataset: MSR-VTT

**Dataset:** MSR-VTT (Microsoft Research Video to Text)  
**Task:** Video captioning (each video paired with natural language description)  
**Fallback:** COCO (Common Objects in Context) for image-caption pairs

**Memory efficiency:**
- `keep_in_memory=False` → dataset memory-mapped from disk
- Only current batch loaded to RAM/GPU during iteration
- Essential for large-scale datasets that don't fit in memory

**Data format:**
- Videos/images converted to frame sequences
- Captions tokenized with fixed length (77 tokens) + padding
- Static images repeated across temporal dimension for V-JEPA2 compatibility

In [ ]:
print('\nLoading dataset...')
# HuggingFace datasets stream from disk (memory-mapped)
# Data only loaded to RAM/GPU batch-by-batch during training
try:
    dataset = load_dataset('AlexZigma/msr-vtt', split='train', keep_in_memory=False)
    print(f'✓ MSR-VTT: {len(dataset)} videos (streaming from disk)')
except:
    print('MSR-VTT unavailable, using COCO...')
    dataset = load_dataset('HuggingFaceM4/COCO', split='train', keep_in_memory=False)
    print(f'✓ COCO: {len(dataset)} images (streaming from disk)')

sample = dataset[0]
print(f'\nKeys: {list(sample.keys())}')
if 'caption' in sample:
    print(f'Caption: {sample["caption"][:80]}...')
elif 'sentences' in sample:
    print(f'Caption: {sample["sentences"]["raw"][0][:80]}...')

## 5. Data Preprocessing

The `VideoTextDataset` handles:

1. **Frame extraction:** Convert images to frame sequences (repeat for static images)
2. **Vision preprocessing:** Apply V-JEPA2's video processor (normalization, resizing)
3. **Text tokenization:** Convert captions to token IDs with padding/truncation
4. **Dimension management:** Remove spurious batch dimensions from processor output

**Key detail:** Video processor returns `[1, T, C, H, W]`, but DataLoader expects `[T, C, H, W]` per sample. We squeeze out the batch dimension before returning, then DataLoader re-batches to `[B, T, C, H, W]`.

In [ ]:
class VideoTextDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, processor, num_frames=16):  # Back to 16
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.processor = processor
        self.num_frames = num_frames
        self.has_image = 'image' in hf_dataset[0]
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        # Load image as repeated frames
        if self.has_image:
            img = sample['image'].convert('RGB')
            frames = [img] * self.num_frames
        else:
            frames = [Image.new('RGB', (224, 224))] * self.num_frames
        
        # Process for vision model
        pixels = self.processor(videos=frames, return_tensors='pt')['pixel_values_videos']
        
        # Remove batch dimension: [1, T, C, H, W] -> [T, C, H, W]
        while pixels.dim() > 4:
            if pixels.shape[0] == 1:
                pixels = pixels.squeeze(0)
            else:
                break
        
        # Get caption
        if 'caption' in sample:
            caption = sample['caption']
        elif 'sentences' in sample:
            caption = sample['sentences']['raw'][0]
        else:
            caption = 'A video.'
        
        # Tokenize
        tokens = self.tokenizer(
            caption,
            return_tensors='pt',
            padding='max_length',
            max_length=77,  # Back to 77
            truncation=True
        )
        
        return {
            'pixels': pixels,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }

train_dataset = VideoTextDataset(dataset, text_tokenizer, vision_processor)
print(f'\n✓ Dataset ready: {len(train_dataset)} samples')

## 6. VL-JEPA Architecture (2-GPU)

### Forward Pass Data Flow

```
Visual Input [B,T,C,H,W]
    ↓
V-JEPA2 Encoder (frozen, GPU 0) → [B,T,vision_dim]
    ↓ (transfer to GPU 1)
Projection Layer → [B,T,predictor_dim]
    ↓
Llama Layers 8-15 (non-causal) → [B,T,predictor_dim]
    ↓
Prediction Head → [B,T,embed_dim]
    ↓
L2 Normalize → predicted_emb

Text Input [B,seq_len]
    ↓
Gemma Encoder (GPU 1) → [B,seq_len,text_dim]
    ↓
Text Projection → [B,seq_len,embed_dim]
    ↓
L2 Normalize → target_emb
```

### Contrastive Loss: InfoNCE

**Objective:** Maximize similarity between corresponding video-text pairs while minimizing similarity to non-corresponding pairs.

**Why InfoNCE?**
- Implicitly learns a shared embedding space for both modalities
- Scales well to large batch sizes (more negatives = better contrastive signal)
- Temperature parameter τ controls distribution sharpness

**Loss computation:**
1. Pool temporal dimension: `pred → [B,embed_dim]`, `target → [B,embed_dim]`
2. Compute similarity matrix: `S = (pred @ target.T) / τ`
3. Treat batch as positive/negative pairs:
   - Diagonal elements = positive pairs (same index = matching video-text)
   - Off-diagonal = negative pairs
4. Bidirectional loss: `L = (L_v2t + L_t2v) / 2`

**Temperature τ:** Lower values make the model more confident (sharper distribution), higher values increase entropy. Learned during training.

### Projection Heads

Both modalities project to shared `embed_dim=2048` space. This dimensionality:
- High enough to preserve semantic information
- Low enough for efficient similarity computation
- Standard for contrastive vision-language models

In [ ]:
class VLJEPA(nn.Module):
    def __init__(self, vision, predictor, proj_in, text, embed_dim=2048):
        super().__init__()
        self.vision = vision  # GPU 0
        self.predictor = predictor  # GPU 1 - full LlamaModel
        self.proj_in = proj_in  # GPU 1
        self.text = text  # GPU 1
        
        self.pred_proj = nn.Linear(predictor_dim, embed_dim).to(device1)
        self.text_proj = nn.Linear(text_dim, embed_dim).to(device1)
        self.temperature = nn.Parameter(torch.tensor(0.07, dtype=torch.float16).to(device1))
    
    def forward(self, pixels, input_ids, attention_mask):
        # Input pixels: [B, T, C, H, W] - V-JEPA2 expects this format
        B, T, C, H, W = pixels.shape
        
        # Encode video on GPU 0 (frozen)
        with torch.no_grad():
            pixels = pixels.to(device0, dtype=torch.float16)
            v_out = self.vision(pixel_values_videos=pixels)
            if hasattr(v_out, 'pooler_output') and v_out.pooler_output is not None:
                visual = v_out.pooler_output
                visual = visual.unsqueeze(1).expand(-1, T, -1)
            else:
                visual = v_out.last_hidden_state
                if visual.dim() == 2:
                    visual = visual.unsqueeze(1).expand(-1, T, -1)
            # Transfer to GPU 1
            visual = visual.to(device1)
        
        # Predictor on GPU 1: [B, T, hidden_dim]
        h = self.proj_in(visual)
        
        # Generate position info and non-causal attention mask
        seq_len = h.shape[1]
        cache_position = torch.arange(seq_len, device=device1)
        
        # Pass through full Llama model (layers 0-7 frozen, 8-15 trainable)
        # LlamaModel expects input_ids but we pass hidden states via inputs_embeds
        predictor_out = self.predictor(
            inputs_embeds=h,
            use_cache=False,
            return_dict=True
        )
        h = predictor_out.last_hidden_state
        
        
        # Pass through predictor layers with position embeddings
        pred = F.normalize(self.pred_proj(h), dim=-1)
        
        # Text encoder on GPU 1
        input_ids = input_ids.to(device1)
        attention_mask = attention_mask.to(device1)
        text_out = self.text(input_ids=input_ids, attention_mask=attention_mask)

        # Debug: check text encoder
        if torch.isnan(text_out.last_hidden_state).any():
            print(f"⚠ NaN in text encoder output!")

        target = self.text_proj(text_out.last_hidden_state)
        if torch.isnan(target).any():
            print(f"⚠ NaN after text_proj!")

        target = F.normalize(target, dim=-1, eps=1e-6)
        if torch.isnan(target).any():
            print(f"⚠ NaN after text normalize!")
        
        return pred, target
    
    def compute_loss(self, pred, target, mask):
        # Pool (all on GPU 1)
        pred_pool = pred.mean(dim=1)
        target_pool = (target * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        pred_pool = F.normalize(pred_pool, dim=-1)
        target_pool = F.normalize(target_pool, dim=-1)
        
        # InfoNCE
        sim = torch.matmul(pred_pool, target_pool.T) / self.temperature
        labels = torch.arange(sim.shape[0], device=device1)
        loss_v2t = F.cross_entropy(sim, labels)
        loss_t2v = F.cross_entropy(sim.T, labels)
        return (loss_v2t + loss_t2v) / 2

model = VLJEPA(vision_model, predictor, predictor_proj_in, text_model)

# Convert trainable parameters to float16 - CRITICAL for dtype consistency
model.pred_proj = model.pred_proj.half()
model.text_proj = model.text_proj.half()
# proj_in already converted to half during initialization
# Predictor layers 8-15 are already float16 from loading
# Only need to ensure projections are half
model.text.half()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M | Frozen: {(total-trainable)/1e6:.1f}M')
print(f'GPU 0: Vision encoder (~304M frozen)')
print(f'GPU 1: Predictor (~490M) + Text encoder (~2048M) + projections')

## 7. Training Configuration

### Memory Optimization Strategies

**Gradient checkpointing:** Trade compute for memory by recomputing activations during backward pass
- Applied to Llama predictor layers and Gemma text encoder
- Reduces activation memory by ~50-70% at cost of ~30% slower training

**Multi-GPU split:** Distribute models across 2 GPUs
- GPU 0: V-JEPA2 (304M frozen)
- GPU 1: Llama predictor (490M) + Gemma (2B) + projections
- Total VRAM split more evenly than single GPU

**Mixed precision (FP16):** All models use float16
- Halves memory usage compared to float32
- Modern GPUs (Ampere+) have dedicated FP16 cores for speed

**Differential learning rates:**
- Base LR: 1e-4 for predictor and projection heads
- Text LR: 5e-6 (0.05× base) for Gemma encoder
- Prevents catastrophic forgetting of pre-trained language knowledge

In [ ]:
# DataLoader - streams data from disk, only loads batches as needed
batch_size = 4
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=False,  # Keep data on CPU, transfer to GPU in forward pass
    drop_last=True
)
print(f'DataLoader: {len(train_loader)} batches (batch_size={batch_size})\n')

# Optimizer (differential learning rates)
base_lr = 1e-4
text_lr = base_lr * 0.05

# Predictor trainable params (layers 8-15)
predictor_params = []
for i in range(8, 16):
    predictor_params.extend(list(model.predictor.layers[i].parameters()))
predictor_params.extend(list(model.pred_proj.parameters()))
predictor_params.extend(list(model.proj_in.parameters()))

# Text encoder params
text_params = list(model.text.parameters()) + list(model.text_proj.parameters())

# Filter for requires_grad
predictor_params = [p for p in predictor_params if p.requires_grad]
text_params = [p for p in text_params if p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': predictor_params, 'lr': base_lr},
    {'params': text_params, 'lr': text_lr}
], weight_decay=0.01)

print(f'Base LR: {base_lr}')
print(f'Text LR: {text_lr}')

## 8. Training Loop

In [ ]:
max_steps = 500
log_interval = 50

print(f'\nTraining for {max_steps} steps...\n')

model.train()
# Keep vision encoder in eval mode (frozen)
model.vision.eval()

global_step = 0

for batch in tqdm(train_loader, total=max_steps):
    if global_step >= max_steps:
        break
    
    # Data starts on CPU, model handles GPU placement
    pixels = batch['pixels']
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    
    # Forward (model handles GPU transfers)
    pred, target = model(pixels, input_ids, attention_mask)
    loss = model.compute_loss(pred, target, attention_mask.to(device1))
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    global_step += 1
    
    if global_step % log_interval == 0:
        print(f'Step {global_step} | Loss: {loss.item():.4f} | Temp: {model.temperature.item():.4f}')

print('\n✓ Training complete!')

## 9. Save Checkpoint

In [ ]:
checkpoint = {
    'predictor_layers': predictor_layers.state_dict(),
    'proj_in': predictor_proj_in.state_dict(),
    'pred_proj': model.pred_proj.state_dict(),
    'text_model': text_model.state_dict(),
    'text_proj': model.text_proj.state_dict(),
    'temperature': model.temperature.data,
    'optimizer': optimizer.state_dict(),
}

torch.save(checkpoint, 'vljepa_checkpoint_2gpu.pt')
print('✓ Saved to vljepa_checkpoint_2gpu.pt')